# DCC 119 Mission 1 — Baseline + 검증용 새 노트북

목표: **신고자(Speaker=1) 음성의 성별(M/F) 분류**

현재 실험 구조:
`신고자 음성(16kHz, mono, 4초) → Log-Mel Spectrogram → CNN → M/F`

이 노트북은 기존 작업을 처음부터 다시 실행할 수 있도록 필요한 코드만 묶은 버전이다.

### 실행 전
- Colab 런타임을 **GPU**로 설정
- `dcc119_m1_subset.zip`을 `/content`에 업로드


In [1]:
# 0. 환경 확인
import os, sys, json, random, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA available: True
CUDA: 12.8
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 1. 필요한 패키지 설치
!pip -q install librosa soundfile tqdm scikit-learn


In [4]:
# 2. subset zip 압축 해제
ZIP_PATH = Path("/content/dcc119_m1_subset.zip")
ROOT = Path("/content/dcc119_m1_subset")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"{ZIP_PATH} 가 없습니다. Colab /content에 dcc119_m1_subset.zip을 업로드하세요."
    )

if not ROOT.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")

print("ROOT:", ROOT)
print("ROOT exists:", ROOT.exists())
print("Train M:", len(list((ROOT/"train"/"M").glob("*.wav"))))
print("Train F:", len(list((ROOT/"train"/"F").glob("*.wav"))))
print("Valid M:", len(list((ROOT/"valid"/"M").glob("*.wav"))))
print("Valid F:", len(list((ROOT/"valid"/"F").glob("*.wav"))))


ROOT: /content/dcc119_m1_subset
ROOT exists: True
Train M: 990
Train F: 1010
Valid M: 501
Valid F: 499


In [5]:
# 3. 메타데이터 확인
TRAIN_META = ROOT / "metadata" / "train.csv"
VALID_META = ROOT / "metadata" / "valid.csv"

train_meta = pd.read_csv(TRAIN_META)
valid_meta = pd.read_csv(VALID_META)

print("Train metadata shape:", train_meta.shape)
print("Valid metadata shape:", valid_meta.shape)
print("\nColumns:", train_meta.columns.tolist())

print("\nTrain gender:")
print(train_meta["gender"].value_counts())

print("\nValid gender:")
print(valid_meta["gender"].value_counts())


Train metadata shape: (2000, 8)
Valid metadata shape: (1000, 8)

Columns: ['sample_id', 'conversation_id', 'original_audio', 'startAt', 'endAt', 'gender', 'output_path', 'utterance_index']

Train gender:
gender
F    1010
M     990
Name: count, dtype: int64

Valid gender:
gender
M    501
F    499
Name: count, dtype: int64


In [6]:
# 4. Log-Mel feature 캐시 생성
import librosa
from tqdm.auto import tqdm

CACHE_DIR = ROOT / "mel_cache"
TRAIN_CACHE = CACHE_DIR / "train"
VALID_CACHE = CACHE_DIR / "valid"
TRAIN_CACHE.mkdir(parents=True, exist_ok=True)
VALID_CACHE.mkdir(parents=True, exist_ok=True)

SR = 16000
N_MELS = 64
N_FFT = 1024
HOP_LENGTH = 256

def make_mel(wav_path):
    y, sr = librosa.load(wav_path, sr=SR, mono=True)
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)

def build_cache(meta_df, split_name, out_dir):
    failures = []
    count = 0

    for _, row in tqdm(
        meta_df.iterrows(), total=len(meta_df), desc=f"cache {split_name}"
    ):
        sample_id = str(row["sample_id"])
        gender = str(row["gender"])
        wav_path = Path(row["output_path"])

        if not wav_path.is_absolute():
            wav_path = ROOT / wav_path

        out_path = out_dir / f"{sample_id}_{gender}.npy"

        if out_path.exists():
            count += 1
            continue

        try:
            feat = make_mel(wav_path)
            np.save(out_path, feat)
            count += 1
        except Exception as e:
            failures.append((sample_id, str(wav_path), repr(e)))

    print(f"{split_name}: cache={count}, failures={len(failures)}")
    if failures:
        print("first failure:", failures[0])
    return failures

train_failures = build_cache(train_meta, "train", TRAIN_CACHE)
valid_failures = build_cache(valid_meta, "valid", VALID_CACHE)


cache train:   0%|          | 0/2000 [00:00<?, ?it/s]

/tmp/ipykernel_1974/2028867975.py:17: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=SR, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


train: cache=0, failures=2000
first failure: ('651e50c72f06ed4a6e31e3d9_utt0010_M', '/content/dcc119_m1_subset/C:\\Users\\hanju\\Downloads\\088.위급상황 음성-음향_고도화_119 지능형 신고접수 음성 인식 데이터\\3.개방데이터\\1.데이터\\dcc119_m1_subset\\train\\M\\651e50c72f06ed4a6e31e3d9_utt0010_M.wav', "FileNotFoundError(2, 'No such file or directory')")


cache valid:   0%|          | 0/1000 [00:00<?, ?it/s]

valid: cache=0, failures=1000
first failure: ('651e4c3e8c3bda5f9fa38ada_utt0018_M', '/content/dcc119_m1_subset/C:\\Users\\hanju\\Downloads\\088.위급상황 음성-음향_고도화_119 지능형 신고접수 음성 인식 데이터\\3.개방데이터\\1.데이터\\dcc119_m1_subset\\valid\\M\\651e4c3e8c3bda5f9fa38ada_utt0018_M.wav', "FileNotFoundError(2, 'No such file or directory')")


In [12]:
# 현재 subset 구조와 wav 파일 존재 여부 확인

from pathlib import Path

ROOT = Path("/content/dcc119_m1_subset")

print("ROOT:", ROOT)
print("ROOT exists:", ROOT.exists())

print("\nTrain F WAV:",
      len(list((ROOT / "train" / "F").glob("*.wav"))))

print("Train M WAV:",
      len(list((ROOT / "train" / "M").glob("*.wav"))))

print("Valid F WAV:",
      len(list((ROOT / "valid" / "F").glob("*.wav"))))

print("Valid M WAV:",
      len(list((ROOT / "valid" / "M").glob("*.wav"))))

print("\nMetadata files:")
print("train.csv:", (ROOT / "metadata" / "train.csv").exists())
print("valid.csv:", (ROOT / "metadata" / "valid.csv").exists())

ROOT: /content/dcc119_m1_subset
ROOT exists: True

Train F WAV: 1010
Train M WAV: 990
Valid F WAV: 499
Valid M WAV: 501

Metadata files:
train.csv: True
valid.csv: True


In [13]:
# Log-Mel cache 새로 생성
import librosa
import numpy as np
from tqdm.auto import tqdm

CACHE_DIR = ROOT / "mel_cache"
TRAIN_CACHE = CACHE_DIR / "train"
VALID_CACHE = CACHE_DIR / "valid"

TRAIN_CACHE.mkdir(parents=True, exist_ok=True)
VALID_CACHE.mkdir(parents=True, exist_ok=True)

SR = 16000
N_MELS = 64
N_FFT = 1024
HOP_LENGTH = 256


def make_mel(wav_path):
    y, sr = librosa.load(
        str(wav_path),
        sr=SR,
        mono=True
    )

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    ).astype(np.float32)

    return mel_db


def build_cache_from_folder(split):
    src_root = ROOT / split
    out_root = CACHE_DIR / split

    wav_files = list(src_root.glob("*/*.wav"))

    print(f"\n[{split}] WAV files:", len(wav_files))

    failures = []

    for wav_path in tqdm(wav_files, desc=f"{split} cache"):
        out_path = out_root / f"{wav_path.stem}.npy"

        if out_path.exists():
            continue

        try:
            mel = make_mel(wav_path)
            np.save(out_path, mel)
        except Exception as e:
            failures.append((wav_path.name, repr(e)))

    print(f"{split} cache 완료")
    print("실패:", len(failures))

    if failures:
        print("첫 번째 실패:", failures[0])


build_cache_from_folder("train")
build_cache_from_folder("valid")


[train] WAV files: 2000


train cache:   0%|          | 0/2000 [00:00<?, ?it/s]

train cache 완료
실패: 0

[valid] WAV files: 1000


valid cache:   0%|          | 0/1000 [00:00<?, ?it/s]

valid cache 완료
실패: 0


In [14]:
# cache 생성 확인

print("Train cache:")
train_cache_files = sorted(TRAIN_CACHE.glob("*.npy"))
for p in train_cache_files[:5]:
    print(p.name)

print("\nTrain cache count:", len(train_cache_files))

print("\nValid cache:")
valid_cache_files = sorted(VALID_CACHE.glob("*.npy"))
for p in valid_cache_files[:5]:
    print(p.name)

print("\nValid cache count:", len(valid_cache_files))

Train cache:
651e464d69a4f266f0626820_utt0017_F.npy
651e464d69a4f266f0626864_utt0036_M.npy
651e464d69a4f266f062687b_utt0014_M.npy
651e464d69a4f266f06268ac_utt0008_M.npy
651e464d69a4f266f06268d1_utt0026_M.npy

Train cache count: 2000

Valid cache:
651e464d69a4f266f062686c_utt0012_M.npy
651e464d69a4f266f062688c_utt0017_F.npy
651e464d69a4f266f06268e5_utt0009_M.npy
651e464d69a4f266f062695c_utt0069_F.npy
651e464d69a4f266f062696a_utt0004_M.npy

Valid cache count: 1000


In [15]:
# 5. CacheDataset + DataLoader
from torch.utils.data import Dataset, DataLoader

LABEL2ID = {"F": 0, "M": 1}
ID2LABEL = {0: "F", 1: "M"}


class MelDataset(Dataset):
    def __init__(self, meta_df, cache_dir):
        self.df = meta_df.reset_index(drop=True).copy()
        self.cache_dir = Path(cache_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        sample_id = str(row["sample_id"])
        gender = str(row["gender"])

        cache_path = self.cache_dir / f"{sample_id}.npy"

        if not cache_path.exists():
            raise FileNotFoundError(
                f"Cache 없음: {cache_path}"
            )

        x = np.load(cache_path).astype(np.float32)

        x = torch.from_numpy(x).unsqueeze(0)

        y = torch.tensor(
            LABEL2ID[gender],
            dtype=torch.long
        )

        return x, y, sample_id


train_ds = MelDataset(
    train_meta,
    TRAIN_CACHE
)

valid_ds = MelDataset(
    valid_meta,
    VALID_CACHE
)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

x0, y0, sid0 = train_ds[0]

print("Sample feature shape:", tuple(x0.shape))
print("Label:", ID2LABEL[int(y0)])
print("Sample ID:", sid0)

Sample feature shape: (1, 64, 251)
Label: M
Sample ID: 651e50c72f06ed4a6e31e3d9_utt0010_M


In [16]:
# 6. Small CNN baseline
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallCNN().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Device:", DEVICE)


Device: cuda


In [17]:
# 7. 학습 + best model 저장
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(x)
            loss = criterion(logits, y)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_count += x.size(0)

    return total_loss / total_count, total_correct / total_count

EPOCHS = 5
best_valid_acc = -1.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss, train_acc = run_epoch(
        model, train_loader, criterion, optimizer
    )
    valid_loss, valid_acc = run_epoch(
        model, valid_loader, criterion, optimizer=None
    )

    elapsed = time.time() - start

    history.append({
        "epoch": epoch,
        "time_sec": elapsed,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
    })

    print(
        f"Epoch {epoch} | {elapsed:.2f}s | "
        f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | "
        f"Valid Loss {valid_loss:.4f} | Valid Acc {valid_acc:.4f}"
    )

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

history_df = pd.DataFrame(history)
print("\nBest Valid Accuracy:", best_valid_acc)
display(history_df)

BEST_CKPT = ROOT / "best_m1_smallcnn.pt"
torch.save(
    {
        "model_state_dict": best_state,
        "best_valid_acc": best_valid_acc,
        "history": history,
    },
    BEST_CKPT,
)
print("Saved:", BEST_CKPT)


Epoch 1 | 4.17s | Train Loss 0.6725 | Train Acc 0.5775 | Valid Loss 0.7466 | Valid Acc 0.5170
Epoch 2 | 2.53s | Train Loss 0.6158 | Train Acc 0.6755 | Valid Loss 0.9112 | Valid Acc 0.5110
Epoch 3 | 2.53s | Train Loss 0.5568 | Train Acc 0.7285 | Valid Loss 0.8900 | Valid Acc 0.5840
Epoch 4 | 2.53s | Train Loss 0.5433 | Train Acc 0.7375 | Valid Loss 0.7512 | Valid Acc 0.6300
Epoch 5 | 2.67s | Train Loss 0.5409 | Train Acc 0.7465 | Valid Loss 0.5273 | Valid Acc 0.7400

Best Valid Accuracy: 0.74


,epoch,time_sec,train_loss,train_acc,valid_loss,valid_acc
0,1,4.168462,0.672547,0.5775,0.746578,0.517
1,2,2.532295,0.615761,0.6755,0.911158,0.511
2,3,2.525373,0.556847,0.7285,0.889977,0.584
3,4,2.526950,0.543332,0.7375,0.751209,0.630
4,5,2.668280,0.540885,0.7465,0.527296,0.740


Saved: /content/dcc119_m1_subset/best_m1_smallcnn.pt


In [18]:
# 8. Best checkpoint 로드 + utterance-level 평가
ckpt = torch.load(BEST_CKPT, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(DEVICE)
model.eval()

rows = []

with torch.no_grad():
    for x, y, sample_ids in valid_loader:
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)

        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred_ids = probs.argmax(axis=1)
        y_np = y.numpy()

        for sid, yi, pi, p in zip(sample_ids, y_np, pred_ids, probs):
            rows.append({
                "sample_id": str(sid),
                "true_id": int(yi),
                "pred_id": int(pi),
                "true_gender": ID2LABEL[int(yi)],
                "pred_gender": ID2LABEL[int(pi)],
                "p_F": float(p[0]),
                "p_M": float(p[1]),
            })

pred_df = pd.DataFrame(rows)
utterance_acc = (pred_df["true_id"] == pred_df["pred_id"]).mean()

print(
    f"Utterance-level Accuracy: "
    f"{utterance_acc:.4f} ({utterance_acc*100:.2f}%)"
)
display(pred_df.head())


Utterance-level Accuracy: 0.7400 (74.00%)


,sample_id,true_id,pred_id,true_gender,pred_gender,p_F,p_M
0,651e4c3e8c3bda5f9fa38ada_utt0018_M,1,1,M,M,0.171939,0.828061
1,651e53b5e498f89fe56782f6_utt0011_M,1,1,M,M,0.334995,0.665005
2,651e53da81ad054c47959aa9_utt0009_F,0,0,F,F,0.956892,0.043108
3,651e551aaa666596ee75fa70_utt0007_M,1,0,M,F,0.799837,0.200163
4,651e49aac10cdef4499a9425_utt0011_M,1,0,M,F,0.697989,0.302011


## 9. ★ 핵심 검증 — Conversation-level 평가

여기서는 `sample_id`를 기준으로 validation metadata의 `conversation_id`를 붙인다.

같은 통화에 속한 신고자 발화를 묶어서 두 가지 방법을 확인한다.

1. **Majority vote**: 개별 발화의 M/F 예측을 다수결
2. **Mean probability**: 통화 내 발화들의 F/M 확률을 평균낸 뒤 선택

주의: 이 결과는 현재 만든 **내부 validation subset의 참고 지표**이다. 대회가 반드시 이 방식으로 최종 평가한다는 뜻은 아니다.


In [19]:
# 10. sample_id -> conversation_id 연결 + leakage 검사
link_df = valid_meta[
    ["sample_id", "conversation_id", "gender", "startAt", "endAt", "utterance_index"]
].copy()

link_df["sample_id"] = link_df["sample_id"].astype(str)
pred_df["sample_id"] = pred_df["sample_id"].astype(str)

eval_df = pred_df.merge(
    link_df,
    on="sample_id",
    how="left",
    validate="one_to_one",
)

print("Missing conversation_id:", eval_df["conversation_id"].isna().sum())
print("Validation conversations:", eval_df["conversation_id"].nunique())
print("Rows:", len(eval_df))

train_conv = set(train_meta["conversation_id"].astype(str))
valid_conv = set(valid_meta["conversation_id"].astype(str))
overlap = train_conv & valid_conv

print("Train/Valid conversation overlap:", len(overlap))
if overlap:
    print("WARNING examples:", list(overlap)[:10])


Missing conversation_id: 0
Validation conversations: 920
Rows: 1000
Train/Valid conversation overlap: 0


In [20]:
# 11. Conversation-level majority vote + mean probability
def majority_label(series):
    counts = series.value_counts()
    return counts.idxmax()

conv_rows = []

for conv_id, g in eval_df.groupby("conversation_id"):
    true_gender = g["gender"].iloc[0]

    majority_pred = majority_label(g["pred_gender"])

    mean_p_f = g["p_F"].mean()
    mean_p_m = g["p_M"].mean()
    mean_prob_pred = "F" if mean_p_f >= mean_p_m else "M"

    counts = g["pred_gender"].value_counts()
    f_count = int(counts.get("F", 0))
    m_count = int(counts.get("M", 0))

    conv_rows.append({
        "conversation_id": str(conv_id),
        "true_gender": true_gender,
        "majority_pred": majority_pred,
        "mean_prob_pred": mean_prob_pred,
        "n_utterances": len(g),
        "F_votes": f_count,
        "M_votes": m_count,
        "mean_p_F": mean_p_f,
        "mean_p_M": mean_p_m,
        "majority_correct": majority_pred == true_gender,
        "mean_prob_correct": mean_prob_pred == true_gender,
    })

conv_df = pd.DataFrame(conv_rows)

majority_acc = conv_df["majority_correct"].mean()
mean_prob_acc = conv_df["mean_prob_correct"].mean()

print(f"Conversation count: {len(conv_df)}")
print(
    f"Majority-vote conversation Accuracy: "
    f"{majority_acc:.4f} ({majority_acc*100:.2f}%)"
)
print(
    f"Mean-probability conversation Accuracy: "
    f"{mean_prob_acc:.4f} ({mean_prob_acc*100:.2f}%)"
)

print("\nUtterances per conversation:")
print(conv_df["n_utterances"].describe())

print(
    "\nMajority ties:",
    int((conv_df["F_votes"] == conv_df["M_votes"]).sum())
)

display(conv_df.head(10))


Conversation count: 920
Majority-vote conversation Accuracy: 0.7457 (74.57%)
Mean-probability conversation Accuracy: 0.7500 (75.00%)

Utterances per conversation:
count    920.000000
mean       1.086957
std        0.296962
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        3.000000
Name: n_utterances, dtype: float64

Majority ties: 22


,conversation_id,true_gender,majority_pred,mean_prob_pred,n_utterances,F_votes,M_votes,mean_p_F,mean_p_M,majority_correct,mean_prob_correct
0,651e464d69a4f266f062686c,M,M,M,1,0,1,0.319259,0.680741,True,True
1,651e464d69a4f266f062688c,F,M,M,1,0,1,0.445858,0.554142,False,False
2,651e464d69a4f266f06268e5,M,M,M,1,0,1,0.427368,0.572632,True,True
3,651e464d69a4f266f062695c,F,M,M,1,0,1,0.233894,0.766106,False,False
4,651e464d69a4f266f062696a,M,M,M,1,0,1,0.317656,0.682344,True,True
5,651e464d69a4f266f0626a9f,F,F,F,1,1,0,0.794104,0.205896,True,True
6,651e464d69a4f266f0626ace,M,F,F,1,1,0,0.584171,0.415829,False,False
7,651e464d69a4f266f0626b5c,F,F,F,1,1,0,0.769806,0.230194,True,True
8,651e464d69a4f266f0626baf,M,M,M,1,0,1,0.163545,0.836455,True,True
9,651e464d69a4f266f0626bbb,M,M,M,1,0,1,0.283206,0.716794,True,True


In [21]:
# 12. 오답 및 통화별 투표 분포 확인
wrong_majority = conv_df[
    ~conv_df["majority_correct"]
].sort_values("n_utterances", ascending=False)

wrong_mean = conv_df[
    ~conv_df["mean_prob_correct"]
].sort_values("n_utterances", ascending=False)

print("Majority-vote wrong conversations:", len(wrong_majority))
if len(wrong_majority):
    display(wrong_majority.head(20))

print("Mean-probability wrong conversations:", len(wrong_mean))
if len(wrong_mean):
    display(wrong_mean.head(20))

print("\nConversation utterance count distribution:")
display(
    conv_df["n_utterances"]
    .value_counts()
    .sort_index()
    .head(30)
    .rename("conversation_count")
    .to_frame()
)


Majority-vote wrong conversations: 234


,conversation_id,true_gender,majority_pred,mean_prob_pred,n_utterances,F_votes,M_votes,mean_p_F,mean_p_M,majority_correct,mean_prob_correct
851,651e54cf6032e904f5dcecfe,M,F,F,3,2,1,0.560751,0.439249,False,False
60,651e498789e4964417c98254,F,M,M,2,0,2,0.405745,0.594255,False,False
99,651e49ec1d10874a0f4ecd27,M,F,M,2,1,1,0.469167,0.530833,False,True
65,651e49aac10cdef4499a9425,M,F,F,2,1,1,0.522856,0.477144,False,False
353,651e4e712e98ee7120968ac4,M,F,M,2,1,1,0.469808,0.530192,False,True
339,651e4d9be84a30cdf982d53d,M,F,M,2,1,1,0.417030,0.582970,False,True
280,651e4c3e8c3bda5f9fa38740,F,M,M,2,1,1,0.455537,0.544463,False,False
182,651e4a8805085d58eeeb1941,M,F,F,2,2,0,0.597505,0.402495,False,False
162,651e4a6ede6495f4e9d36b9b,M,F,F,2,2,0,0.610838,0.389162,False,False
40,651e496f5d60e22224166f1e,F,M,M,2,1,1,0.494012,0.505988,False,False


Mean-probability wrong conversations: 230


,conversation_id,true_gender,majority_pred,mean_prob_pred,n_utterances,F_votes,M_votes,mean_p_F,mean_p_M,majority_correct,mean_prob_correct
851,651e54cf6032e904f5dcecfe,M,F,F,3,2,1,0.560751,0.439249,False,False
65,651e49aac10cdef4499a9425,M,F,F,2,1,1,0.522856,0.477144,False,False
102,651e49ec1d10874a0f4ece3c,M,M,F,2,1,1,0.565087,0.434913,True,False
60,651e498789e4964417c98254,F,M,M,2,0,2,0.405745,0.594255,False,False
143,651e4a3786dc055ca8bf486f,M,F,F,2,2,0,0.738550,0.261450,False,False
40,651e496f5d60e22224166f1e,F,M,M,2,1,1,0.494012,0.505988,False,False
278,651e4c3e8c3bda5f9fa38670,M,M,F,2,1,1,0.569947,0.430053,True,False
182,651e4a8805085d58eeeb1941,M,F,F,2,2,0,0.597505,0.402495,False,False
280,651e4c3e8c3bda5f9fa38740,F,M,M,2,1,1,0.455537,0.544463,False,False
331,651e4d9be84a30cdf982d285,M,F,F,2,2,0,0.716680,0.283320,False,False



Conversation utterance count distribution:


,conversation_count
n_utterances,
1,844
2,72
3,4


In [22]:
# 13. 최종 요약 — 이 셀 출력 결과를 보내주면 다음 실험을 정할 수 있음
print("===== Mission 1 현재 실험 요약 =====")
print(f"Train samples: {len(train_meta):,}")
print(f"Valid utterances: {len(pred_df):,}")
print(f"Valid conversations: {len(conv_df):,}")
print(f"Best utterance Accuracy: {utterance_acc:.4f} ({utterance_acc*100:.2f}%)")
print(f"Majority conversation Accuracy: {majority_acc:.4f} ({majority_acc*100:.2f}%)")
print(f"Mean-probability conversation Accuracy: {mean_prob_acc:.4f} ({mean_prob_acc*100:.2f}%)")
print(f"Train/Valid conversation overlap: {len(overlap)}")
print(f"Majority ties: {int((conv_df['F_votes'] == conv_df['M_votes']).sum())}")


===== Mission 1 현재 실험 요약 =====
Train samples: 2,000
Valid utterances: 1,000
Valid conversations: 920
Best utterance Accuracy: 0.7400 (74.00%)
Majority conversation Accuracy: 0.7457 (74.57%)
Mean-probability conversation Accuracy: 0.7500 (75.00%)
Train/Valid conversation overlap: 0
Majority ties: 22


### 다음 개발 순서
1. 현재 conversation-level 결과가 왜 나오는지 검증
2. 데이터 규모 확대
3. 4초 vs 6초/8초 비교
4. Small CNN vs ResNet18 등 비교
5. 최종 `inference.py` 작성
